# Laboratório 4 - Visão Computacional

Autores:

- Kayky de Brito dos Santos
- André Marques da Silva
- Rafael de Souza Coelho

Equipe 8 - "Sem Título"

Data de realização dos experimentos: 29 de junho de 2026

Data de publicação do relatório: 5 de julho de 2026

> **Relatório completo:** a nossa execução (parâmetros de calibração estimados, matrizes K/R/t/dist, detecção dos cantos e correção de distorção) e a análise e discussão estão detalhadas no post:
>
> <https://kaykyb.github.io/ufabc-cv/posts/lab4/>
>
> Este notebook reúne os programas OpenCV do laboratório. A execução comentada dos experimentos e a análise dos resultados ficam no post acima.

## Parte 2: Calibração de câmeras

### (A) e (B) Captura das imagens do padrão de calibração

O programa `L4_chess.py` abre a webcam, mostra o vídeo e salva uma imagem do tabuleiro a cada tecla `s` pressionada (saindo com `q`). Ajustamos o nome do arquivo de saída para um integrante da equipe (Rafael) e capturamos 15 imagens.

In [ ]:
import numpy as np
import cv2 as cv

cap = cv.VideoCapture(0)

if not cap.isOpened():
    print("Cannot open camera")
    exit()

i = 0

while True:
    # Captura quadro a quadro
    ret, frame = cap.read()
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break

    cv.imshow('frame', frame)

    # Salva na tecla 's' ou sai na tecla 'q'
    k = cv.waitKey(1)
    if k == ord('s'):
        cv.imwrite("rafael_" + str(i) + ".jpg", frame)
        i = i + 1
        print("frame", i)
    elif k == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

### Calibração com `L4_cal.py`

O programa percorre todas as imagens `.jpg` da pasta, detecta os 6x8 cantos internos do tabuleiro (`findChessboardCorners` + refinamento sub-pixel com `cornerSubPix`) e calibra a câmera com `calibrateCamera`, imprimindo a matriz `K`, o vetor `dist` e os `rvecs`/`tvecs` de cada pose. O número de *features* é `(6, 8)` conforme a observação do enunciado.

In [ ]:
import cv2
import numpy as np
import os
import glob

# Dimensões do tabuleiro (cantos internos)
CHECKERBOARD = (6, 8)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Vetores de pontos 3D (mundo) e 2D (imagem) por tabuleiro
objpoints = []
imgpoints = []

# Coordenadas do mundo para os pontos 3D
objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[0, :, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

images = glob.glob('*.jpg')
for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Encontra os cantos do tabuleiro
    ret, corners = cv2.findChessboardCorners(
        gray, CHECKERBOARD,
        cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)

    # Se detectou, refina os cantos e os exibe
    if ret == True:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)
        img = cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)

    cv2.imshow('img', img)
    cv2.waitKey(0)

cv2.destroyAllWindows()

h, w = img.shape[:2]

# Calibração: usa os pontos 3D conhecidos e os cantos 2D detectados
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None)

print("Camera matrix (K):\n", mtx)
print("dist:\n", dist)
print("rvecs:\n", rvecs)
print("tvecs:\n", tvecs)

A matriz de rotação `R` 3x3 de cada pose é obtida convertendo o `rvec` (vetor de Rodrigues) correspondente:

In [ ]:
# Converte o vetor de Rodrigues da primeira pose na matriz de rotação R 3x3
R0, _ = cv2.Rodrigues(rvecs[0])
print("R (primeira imagem):\n", R0)
print("t (primeira imagem):\n", tvecs[0])

### (D) Correção da distorção radial

Usando os parâmetros da câmera calibrada, refinamos a matriz com `getOptimalNewCameraMatrix` e corrigimos a distorção pelos dois métodos do OpenCV: `undistort()` direto e o *remapping* com `remap()`. Carregue aqui os parâmetros `K` e `dist` obtidos na calibração da câmera (item B).

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Parâmetros de calibração da webcam (item B)
K = np.array([[686.34216058, 0.0, 305.46924272],
              [0.0, 684.21246902, 234.33702301],
              [0.0, 0.0, 1.0]])
dist = np.array([0.0606493768, -0.554142960, -0.00277815433, 0.000999229668, 2.17306000])

img = cv.imread('rafael_0.jpg')
h, w = img.shape[:2]

# Refina a matriz da câmera (alpha=1 mantém todos os pixels)
newK, roi = cv.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))
print("Nova matriz da câmera:\n", newK)
print("ROI válida:", roi)

# Método 1: undistort direto
undistorted = cv.undistort(img, K, dist, None, newK)

# Método 2: remapping
mapx, mapy = cv.initUndistortRectifyMap(K, dist, None, newK, (w, h), 5)
remapped = cv.remap(img, mapx, mapy, cv.INTER_LINEAR)

# Exibe original x corrigidas lado a lado
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
for a, im, t in zip(ax, [img, undistorted, remapped], ['Original', 'undistort()', 'remap()']):
    a.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
    a.set_title(t)
    a.axis('off')
plt.tight_layout()
plt.show()